In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import cv2
import time
import pandas as pd
import numpy as np
import os

# --- CONFIGURATION ---
CSV_PATH = "../../f2_gps.csv"
VIDEO_PATH = "../../../videos/f2_c2_HR.MP4"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
K_CLUSTERS = 64
TOTAL_IMAGES = 30312
IMG_SIZE_GLOBAL = (224, 224)
LOCAL_KPTS_CONFIGS = [3000, 1500, 1000]

GLOBAL_MODELS = {
    "MobileNetV4": "mobilenetv4_conv_small",
    "MobileNetV3": "mobilenetv3_small_100",
    "MobileNetV2": "mobilenetv2_100",
    "ResNet18": "resnet18"
}

class NetVLAD(nn.Module):
    def __init__(self, feature_dim, clusters=64):
        super().__init__()
        self.feature_dim = feature_dim
        self.clusters = clusters
        self.conv = nn.Conv2d(feature_dim, clusters, kernel_size=1)
        self.centroids = nn.Parameter(torch.randn(clusters, feature_dim))

    def forward(self, x):
        N, D, H, W = x.shape
        # x: [N, D, H, W] -> soft_assign: [N, C, H*W]
        soft_assign = F.softmax(self.conv(x).view(N, self.clusters, -1), dim=1)
        
        # x_flatten: [N, D, H*W] -> [N, H*W, D]
        x_flatten = x.view(N, D, -1).permute(0, 2, 1)
        
        vlad = torch.zeros([N, self.clusters, D], device=x.device)
        for k in range(self.clusters):
            # residual: [N, H*W, D]
            residual = x_flatten - self.centroids[k:k+1, :]
            # soft_assign[:, k:k+1, :]: [N, 1, H*W]
            # Reshape for element-wise: [N, H*W, 1]
            a = soft_assign[:, k:k+1, :].permute(0, 2, 1)
            vlad[:, k:k+1, :] = torch.sum(a * residual, dim=1, keepdim=True)
            
        vlad = F.normalize(vlad.view(N, -1), p=2, dim=1)
        return vlad

def get_8_augmentations(tensor):
    aug_list = []
    for flip in [False, True]:
        img = torch.flip(tensor, dims=[3]) if flip else tensor
        for k in range(4):
            aug_list.append(torch.rot90(img, k, dims=[2, 3]))
    return torch.cat(aug_list, dim=0)

def benchmark_pipeline():
    if not os.path.exists(CSV_PATH) or not os.path.exists(VIDEO_PATH):
        print("Check file paths.")
        return

    df = pd.read_csv(CSV_PATH)
    target_frames = df['c2_frame'].dropna().unique()[:500].astype(int)
    cap = cv2.VideoCapture(VIDEO_PATH)
    
    global_results = []
    local_results = []

    # --- 1. GLOBAL FEATURE EXTRACTION ---
    print("--- Benchmarking Global Features (C2, Orientation-Invariant) ---")
    
    for name, model_id in GLOBAL_MODELS.items():
        # Using features_only=True to get conv maps for NetVLAD
        backbone = timm.create_model(model_id, pretrained=True, features_only=True).to(DEVICE).eval()
        
        # Get feature dimension from the final conv layer
        with torch.no_grad():
            dummy_out = backbone(torch.randn(1, 3, 224, 224).to(DEVICE))
            feat_dim = dummy_out[-1].shape[1]
        
        nv_layer = NetVLAD(feat_dim, K_CLUSTERS).to(DEVICE).eval()

        for pool_type in ["GAP", "NetVLAD"]:
            latencies = []
            with torch.no_grad():
                for frame_idx in target_frames:
                    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
                    ret, frame = cap.read()
                    if not ret: continue
                    
                    t0 = time.time()
                    # Step 1: Data Prep
                    img = cv2.resize(frame, IMG_SIZE_GLOBAL)
                    img_t = torch.from_numpy(img).permute(2, 0, 1).float().unsqueeze(0).to(DEVICE) / 255.0
                    
                    # Step 2: Augmentation (8 versions)
                    batch = get_8_augmentations(img_t)
                    
                    # Step 3: Extraction & Aggregation
                    maps = backbone(batch)[-1]
                    
                    if pool_type == "GAP":
                        vecs = F.adaptive_avg_pool2d(maps, (1, 1)).flatten(1)
                    else:
                        vecs = nv_layer(maps)
                    
                    # Step 4: Normalization for Cosine Similarity
                    final_vec = torch.mean(vecs, dim=0, keepdim=True)
                    final_vec = F.normalize(final_vec, p=2, dim=1)
                    
                    latencies.append(time.time() - t0)

            avg_t = np.mean(latencies)
            out_dim = feat_dim if pool_type == "GAP" else feat_dim * K_CLUSTERS
            size_mb = (out_dim * 4 * TOTAL_IMAGES) / (1024**2)
            global_results.append([name, pool_type, f"{avg_t:.4f}", f"{1/avg_t:.2f}", out_dim, f"{size_mb:.2f}"])

    # --- 2. LOCAL FEATURE EXTRACTION (ORB) ---
    print("\n--- Benchmarking Local Features (C2, Half-Resolution) ---")
    
    for kpts in LOCAL_KPTS_CONFIGS:
        orb = cv2.ORB_create(nfeatures=kpts)
        timings = []
        
        for frame_idx in target_frames:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if not ret: continue

            # Time Resize
            t_res_start = time.time()
            h, w = frame.shape[:2]
            resized = cv2.resize(frame, (w // 2, h // 2))
            t_res = (time.time() - t_res_start) * 1000

            # Time ORB
            t_orb_start = time.time()
            kp, des = orb.detectAndCompute(resized, None)
            t_orb = (time.time() - t_orb_start) * 1000
            
            timings.append([t_res, t_orb])

        avg_timings = np.mean(timings, axis=0)
        total_size_gb = (kpts * 32 * TOTAL_IMAGES) / (1024**3)
        local_results.append(["C2", kpts, f"{avg_timings[0]:.2f}", f"{avg_timings[1]:.2f}", f"{total_size_gb:.2f}"])

    cap.release()

    print("\nTABLE 4.1: GLOBAL PERFORMANCE")
    print(pd.DataFrame(global_results, columns=["Model", "Pool", "Time(s)", "Emb/s", "Dim", "Size (MB)"]).to_string(index=False))
    
    print("\nTABLE 4.2: ORB PERFORMANCE")
    print(pd.DataFrame(local_results, columns=["Cam", "Kpts", "Resize(ms)", "ORB(ms)", "Total GB"]).to_string(index=False))

if __name__ == "__main__":
    benchmark_pipeline()

In [ ]:
import numpy as np
import pandas as pd
import time
from math import radians, sin, cos, sqrt, atan2, tan
import matplotlib.pyplot as plt

# Load flight data
df = pd.read_csv(FLIGHT_DATA_CSV)
print(f"Total samples: {len(df)}")

IMAGE_WIDTH = 3840
IMAGE_HEIGHT = 2160
fx = 1796.32
fy = 1797.22

class DropperEvaluator:
    def __init__(self, df):
        self.df = df
        self.results = {}

    def haversine_distance(self, lat1, lon1, lat2, lon2):
        R = 6371000
        lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
        c = 2 * atan2(sqrt(a), sqrt(1-a))
        return R * c

    def compute_footprint(self, altitude):
        width = altitude * IMAGE_WIDTH / fx
        height = altitude * IMAGE_HEIGHT / fy
        return width * height

    def compute_overlap(self, alt1, alt2, lat1, lon1, lat2, lon2):
        area1 = self.compute_footprint(alt1)
        area2 = self.compute_footprint(alt2)
        distance = self.haversine_distance(lat1, lon1, lat2, lon2)
        avg_footprint_radius = sqrt((area1 + area2) / (2 * np.pi))
        if distance >= 2 * avg_footprint_radius:
            return 0.0
        else:
            overlap_ratio = max(0, 1 - (distance / (2 * avg_footprint_radius)))
            return overlap_ratio

    def fixed_interval_dropper(self, interval_seconds=5.0):
        kept_indices = []
        latencies = []
        last_time = self.df['TIME'].iloc[0]
        kept_indices.append(0)
        for i in range(1, len(self.df)):
            start_time = time.time()
            current_time = self.df['TIME'].iloc[i]
            delta_t = current_time - last_time
            if delta_t >= interval_seconds:
                kept_indices.append(i)
                last_time = current_time
            latencies.append(time.time() - start_time)
        return kept_indices, latencies

    def haversine_dropper(self, min_distance_meters=50.0):
        kept_indices = []
        latencies = []
        last_lat = self.df['LATITUDE'].iloc[0]
        last_lon = self.df['LONGITUDE'].iloc[0]
        kept_indices.append(0)
        for i in range(1, len(self.df)):
            start_time = time.time()
            current_lat = self.df['LATITUDE'].iloc[i]
            current_lon = self.df['LONGITUDE'].iloc[i]
            distance = self.haversine_distance(last_lat, last_lon, current_lat, current_lon)
            if distance >= min_distance_meters:
                kept_indices.append(i)
                last_lat = current_lat
                last_lon = current_lon
            latencies.append(time.time() - start_time)
        return kept_indices, latencies

    def area_based_dropper(self, max_overlap_ratio=0.3):
        kept_indices = []
        latencies = []
        last_lat = self.df['LATITUDE'].iloc[0]
        last_lon = self.df['LONGITUDE'].iloc[0]
        last_alt = self.df['ALTITUDE'].iloc[0]
        kept_indices.append(0)
        for i in range(1, len(self.df)):
            start_time = time.time()
            current_lat = self.df['LATITUDE'].iloc[i]
            current_lon = self.df['LONGITUDE'].iloc[i]
            current_alt = self.df['ALTITUDE'].iloc[i]
            overlap = self.compute_overlap(last_alt, current_alt, last_lat, last_lon, current_lat, current_lon)
            if overlap <= max_overlap_ratio:
                kept_indices.append(i)
                last_lat = current_lat
                last_lon = current_lon
                last_alt = current_alt
            latencies.append(time.time() - start_time)
        return kept_indices, latencies

    def evaluate_all(self, interval=5.0, distance=50.0, overlap=0.3):
        total_samples = len(self.df)
        kept_fi, lat_fi = self.fixed_interval_dropper(interval)
        retention_fi = (len(kept_fi) / total_samples) * 100
        avg_lat_fi = np.mean(lat_fi) * 1000
        print(f"Fixed Interval (T={interval} s): Retention {retention_fi:.2f}%, Latency {avg_lat_fi:.4f} ms")

        kept_hv, lat_hv = self.haversine_dropper(distance)
        retention_hv = (len(kept_hv) / total_samples) * 100
        avg_lat_hv = np.mean(lat_hv) * 1000
        print(f"Haversine (r={distance} m): Retention {retention_hv:.2f}%, Latency {avg_lat_hv:.4f} ms")

        kept_ab, lat_ab = self.area_based_dropper(overlap)
        retention_ab = (len(kept_ab) / total_samples) * 100
        avg_lat_ab = np.mean(lat_ab) * 1000
        print(f"Area-Based (α={overlap}): Retention {retention_ab:.2f}%, Latency {avg_lat_ab:.4f} ms")

        self.results = {
            'Fixed Interval': {'retention': retention_fi, 'latency': avg_lat_fi, 'kept': kept_fi},
            'Haversine': {'retention': retention_hv, 'latency': avg_lat_hv, 'kept': kept_hv},
            'Area-Based': {'retention': retention_ab, 'latency': avg_lat_ab, 'kept': kept_ab}
        }
        return self.results


evaluator = DropperEvaluator(df)
intervals = [1, 5, 10, 20, 30, 45]
distances = [10, 50, 100, 200, 300, 800]
overlaps = [0.1, 0.25, 0.3, 0.5, 0.75, 1]

print("Fixed Interval:")
for t in intervals:
    kept, _ = evaluator.fixed_interval_dropper(t)
    print(f"T={t}s: {(len(kept)/len(df)*100):.1f}% retained")

print("Haversine:")
for d in distances:
    kept, _ = evaluator.haversine_dropper(d)
    print(f"r={d}m: {(len(kept)/len(df)*100):.1f}% retained")

print("Area Overlap:")
for o in overlaps:
    kept, _ = evaluator.area_based_dropper(o)
    print(f"alpha={o}: {(len(kept)/len(df)*100):.1f}% retained")